# 解答① ReAct エージェントの部品を作る

> **講師用**: 演習 `ex_01_react_agent.ipynb` の解答です。参加者には配布しません（`handson.sh` は `solutions/` を配らない）。

---

## セットアップ

In [ ]:
# 実行時間: 数秒
import os
os.environ.setdefault('ANONYMIZED_TELEMETRY', 'False')

import sys
import json
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 第3章の共通部品（chapter3/agent_kit.py）を読み込めるようにする
CH3_DIR = (Path('..') if Path('../agent_kit.py').exists() else Path('chapter3')).resolve()
sys.path.insert(0, str(CH3_DIR))
from agent_kit import LocalChatModel, TraceHandler, parse_tool_calls, run_python as _run_python, workspace_path

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

print('キャッシュ先:', os.environ.get('HF_HOME', '~/.cache/huggingface'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')

# エージェントがファイルを読み書きする作業ディレクトリ（chapter3/workspace）
WORKSPACE = CH3_DIR / 'workspace'
WORKSPACE.mkdir(exist_ok=True)
print(f'作業ディレクトリ: {WORKSPACE}')

In [ ]:
# 実行時間: 約2〜3分
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if device == 'cuda':
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto')
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.eval()
print(f'{MODEL_NAME} を読み込みました（モデルのメモリ: {model.get_memory_footprint() / 1024**3:.1f} GB）')

## 課題 1: ツールを作る

In [ ]:
@tool
def count_chars(text: str) -> int:
    """文章の文字数を、空白と改行を除いて数える。文字数の制限を確かめるときに使う。"""
    return sum(not c.isspace() for c in text)


schema = convert_to_openai_tool(count_chars)
print(json.dumps(schema, ensure_ascii=False, indent=2))

# ── 自己チェック ──
fn = schema['function']
assert fn['name'] == 'count_chars'
assert len(fn['description']) >= 10, 'docstring に、何をするツールかを書いてください'
assert fn['parameters']['properties']['text']['type'] == 'string'
assert count_chars.invoke({'text': 'こんにちは 世界\n'}) == 7, '空白と改行は数えません'
print('✅ OK')

## 課題 2: モデルの出力からツール呼び出しを取り出す

In [ ]:
import re


def extract_tool_calls(text: str) -> list[tuple[str, dict]]:
    calls = []
    for raw in re.findall(r'<tool_call>\s*(.*?)\s*</tool_call>', text, re.S):
        obj = json.loads(raw)
        calls.append((obj['name'], obj['arguments']))
    return calls


sample = ('調べてから計算します。\n'
          '<tool_call>\n{"name": "search_docs", "arguments": {"query": "メモリ 上限"}}\n</tool_call>\n'
          '<tool_call>\n{"name": "run_python", "arguments": {"code": "print(2 ** 10)"}}\n</tool_call>')
print(extract_tool_calls(sample))

# ── 自己チェック ──
assert extract_tool_calls(sample) == [('search_docs', {'query': 'メモリ 上限'}), ('run_python', {'code': 'print(2 ** 10)'})]
assert extract_tool_calls('ツールは使いません。答えは 42 です。') == []
print('✅ OK')

## 課題 3: ReAct のグラフを組み立てる

In [ ]:
@tool
def python_exec(code: str) -> str:
    """Python のコードを実行し、print() で表示された内容を返す。計算をするときに使う。"""
    return _run_python(code, WORKSPACE)


tools = [count_chars, python_exec]
llm_with_tools = LocalChatModel(model=model, tokenizer=tokenizer).bind_tools(tools)


def agent_node(state: MessagesState) -> dict:
    return {'messages': [llm_with_tools.invoke(state['messages'])]}


builder = StateGraph(MessagesState)
builder.add_node('agent', agent_node)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', tools_condition)
builder.add_edge('tools', 'agent')

graph = builder.compile()
print(graph.get_graph().draw_ascii())

# ── 自己チェック ──
g = graph.get_graph()
assert set(g.nodes) == {'__start__', 'agent', 'tools', '__end__'}, set(g.nodes)
edges = {(e.source, e.target, e.conditional) for e in g.edges}
assert ('__start__', 'agent', False) in edges
assert ('tools', 'agent', False) in edges
assert ('agent', 'tools', True) in edges and ('agent', '__end__', True) in edges, 'agent からの分かれ道（tools_condition）がありません'
print('✅ OK')

## 課題 4: トレースからコストを集計する

In [ ]:
trace = TraceHandler()
result = graph.invoke(
    {'messages': [HumanMessage('「エージェントとは LLM とループとツールである」の文字数を数えてください。')]},
    config={'callbacks': [trace], 'recursion_limit': 8},
)
print(f'回答: {result["messages"][-1].content}\n')

n_llm = sum(r['kind'] == 'LLM' for r in trace.records)
n_tool = sum(r['kind'] == 'ツール' for r in trace.records)
input_tokens = sum(r['input_tokens'] for r in trace.records if r['kind'] == 'LLM')

print(f'LLM {n_llm} 回、ツール {n_tool} 回、入力トークン {input_tokens}')

# ── 自己チェック ──
t = trace.totals()
assert n_llm == t['LLM の呼び出し回数'] and n_llm >= 1
assert n_tool == t['ツールの呼び出し回数']
assert input_tokens == t['入力トークン数']
print('✅ OK')